# 1. Imports

In [18]:
import pandas as pd
from pathlib import Path
import os

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

pd.set_option('display.max_columns', None)

# 2. Funções

## 2.1. Função para pegar os eventos de uma temporada nos arquivos parquet

In [19]:
def get_season_events_parquet_file_paths(events_competition_season_folder_path):
    
    season_events_parquet_file_paths = [
        str(Path(events_competition_season_folder_path) / season_event_parquet_file) 
        for season_event_parquet_file in os.listdir(events_competition_season_folder_path) 
        if season_event_parquet_file.endswith('.parquet')
        ]
    
    return season_events_parquet_file_paths

# 3. Preparação dos dados

## 3.1. Criação da Sessão Spark

In [20]:
# Criação da sessão Spark local
spark = SparkSession.builder.master("local[*]").appName("season_database").getOrCreate()

## 3.2. Criação do df para pegar os eventos de todas as partidas da temporada de 2022-2023 da Premier League

(dps pode ser interessante levar a parte do schema dos jogadores e da bola p etapa de extração)

In [21]:
events_competition_season_folder_path = str(Path().resolve().parent.parent / "data" / "events" / "1" / "2022-2023")

season_events_parquet_file_paths = get_season_events_parquet_file_paths(events_competition_season_folder_path)

# Criação do dataframe concatenando todos os arquivos parquet dos eventos das partidas entre as temporadas de todas as competições
df_events = spark.read.parquet(*season_events_parquet_file_paths)

df_events = df_events.withColumnsRenamed({
    "id": "eventId",
    "player.id": "eventPlayer.id",
    "player.name": "eventPlayer.name",
    "team.id": "eventTeam.id",
    "team.name": "eventTeam.name",
})

# Schema em Pyspark para poder parsear o json dos dados de tracking dos jogadores que está como string
players_schema = ArrayType(
    StructType([
        #StructField("speed", FloatType(), True),
        StructField("y", FloatType(), True),
        StructField("x", FloatType(), True),
        StructField("player", StructType([
            StructField("id", IntegerType(), True), 
            StructField("name", StringType(), True)]), 
            True),
        StructField("visibility", StringType(), True),
        StructField("confidence", StringType(), True),
        #StructField("jerseyNum", StringType(), True)      
    ])
)

# Schema em Pyspark para poder parsear o json dos dados de tracking da bola que está como string
balls_schema = ArrayType(
    StructType([
        StructField("z", FloatType(), True),
        StructField("y", FloatType(), True),
        StructField("x", FloatType(), True),
        StructField("visibility", StringType(), True)
    ])
)

details_schema = MapType(StringType(), StringType())

df_events = df_events.withColumns({
    # Cria coluna com json parseado para Lista de dicionários para dados de tracking do time mandante
    "homePlayers_parsed": F.from_json("homePlayers", players_schema),
    
    # Cria coluna com json parseado para Lista de dicionários para dados de tracking do time adversário
    "awayPlayers_parsed": F.from_json("awayPlayers", players_schema),

    # Cria coluna com json parseado para dicionário para dados de tracking da bola
    "balls_parsed": F.from_json("balls", balls_schema),

    "details_parsed": F.from_json("details", details_schema)

}).drop('homePlayers', 'awayPlayers', 'balls', 'details')

df_events = df_events.select(
    'competitionId',
    'season', # dps mudar pra seasonId se necessário
    'gameId',
    'eventId',
    'eventType',
    'eventTypeDescription',
    'period',
    'periodDescription',
    'startFormattedGameClock',
    'startGameClock',
    'details_parsed',
    'homeTeam',
    F.col('`eventPlayer.id`').alias('eventPlayerId'),
    F.col('`eventPlayer.name`').alias('eventPlayerName'),
    F.col('`eventTeam.id`').alias('eventTeamId'), 
    F.col('`eventTeam.name`').alias('eventTeamName'), 
    'homePlayers_parsed', 
    'awayPlayers_parsed', 
    'balls_parsed'
)

In [22]:
print('Quantidade de linhas:', df_events.count())

Quantidade de linhas: 945154


## 3.2. Obter jogos da temporada e ajustar identificação do mandante/adversário

(dps pode ser interessante levar isso p etapa de extração)

In [23]:
games_path = str(Path().resolve().parent.parent / "data" / "games.csv")

df_games = spark.read.csv(games_path, header=True)

df_games_raw = df_games.withColumnRenamed("id","gameId").filter(F.col('season') == '2022-2023')

# se venueType == TEAM_HOME, (homeTeam.id == team.id e homeTeam.name == team.name) e (opponentTeam.id == opponentTeam.id e opponentTeam.name == opponentTeam.name)
# se venueType == OPPONENT_HOME, (homeTeam.id == opponentTeam.id e homeTeam.name == opponentTeam.name) e (opponentTeam.id == team.id e opponentTeam.name == team.name)
df_games = (
    df_games_raw.withColumns({
    "homeTeamId": F.when(F.col('venueType') == 'TEAM_HOME', F.col('`team.id`')).otherwise(F.col('`opponentTeam.id`')),
    "homeTeamName": F.when(F.col('venueType') == 'TEAM_HOME', F.col('`team.name`')).otherwise(F.col('`opponentTeam.name`')),

    "opponentTeamId": F.when(F.col('venueType') == 'TEAM_HOME', F.col('`opponentTeam.id`')).otherwise(F.col('`opponentTeam.id`')),
    "opponentTeamName": F.when(F.col('venueType') == 'TEAM_HOME', F.col('`opponentTeam.name`')).otherwise(F.col('`opponentTeam.name`')),
    }).select(
        'gameId', 
        'date',
        'season',
        F.col('`competition.id`').alias('competitionId'),
        F.col('`competition.name`').alias('competitionName'),
        'homeTeamId',
        'homeTeamName',
        'opponentTeamId',
        'opponentTeamName',
        F.col('teamExtraTimeStartSide').alias('homeTeamExtraTimeStartSide'), 
        F.col('teamStartSide').alias('homeTeamStartSide'),
        F.col('`stadium.name`').alias('stadiumName'), 
        F.col('`stadium.length`').cast("float").alias('stadiumLength'), 
        F.col('`stadium.width`').cast("float").alias('stadiumWidth')
    )
)

df_games.show(5)

+------+----------+---------+-------------+---------------+----------+--------------------+--------------+--------------------+--------------------------+-----------------+-------------+-------------+------------+
|gameId|      date|   season|competitionId|competitionName|homeTeamId|        homeTeamName|opponentTeamId|    opponentTeamName|homeTeamExtraTimeStartSide|homeTeamStartSide|  stadiumName|stadiumLength|stadiumWidth|
+------+----------+---------+-------------+---------------+----------+--------------------+--------------+--------------------+--------------------------+-----------------+-------------+-------------+------------+
|  4447|2022-08-13|2022-2023|            1| Premier League|         3|         Aston Villa|             8|             Everton|                     Right|             Left|   Villa Park|        105.0|        68.0|
|  4760|2023-04-25|2022-2023|            1| Premier League|        20|Wolverhampton Wan...|            20|Wolverhampton Wan...|                 

## 4. Junção dos dados dos Jogos + Eventos em uma tabela

In [24]:
df_games_events = df_events.join(df_games.drop('season', 'competitionId', 'competitionName'), on = "gameId", how='left')

df_games_events = (
    df_games_events
    .withColumn(
        'possessionTeam',
        F.when(F.col('homeTeam'), "home")
    )
    .withColumn(
        'homeTeamAttackDirection',
            F.when(
                ((F.col('period') == 1) & (F.col('homeTeamStartSide') == 'Right')) | 
                ((F.col('period') == 2) & (F.col('homeTeamStartSide') == 'Left')), 
                'Left'
            )
            .when(
                ((F.col('period') == 1) & (F.col('homeTeamStartSide') == 'Left')) | 
                ((F.col('period') == 2) & (F.col('homeTeamStartSide') == 'Right')), 
                'Right'
            )
        )
    .withColumn(
            'awayTeamAttackDirection',
            F.when(F.col('homeTeamAttackDirection') == 'Right', 'Left')
            .when(F.col('homeTeamAttackDirection') == 'Left', 'Right')
        )
)

df_games_events.show(5)

+------+-------------+---------+--------------------+------------+--------------------+------+-----------------+-----------------------+--------------+--------------------+--------+-------------+---------------+-----------+-------------+--------------------+--------------------+--------------------+----------+----------+---------------+--------------+----------------+--------------------------+-----------------+----------------+-------------+------------+--------------+-----------------------+-----------------------+
|gameId|competitionId|   season|             eventId|   eventType|eventTypeDescription|period|periodDescription|startFormattedGameClock|startGameClock|      details_parsed|homeTeam|eventPlayerId|eventPlayerName|eventTeamId|eventTeamName|  homePlayers_parsed|  awayPlayers_parsed|        balls_parsed|      date|homeTeamId|   homeTeamName|opponentTeamId|opponentTeamName|homeTeamExtraTimeStartSide|homeTeamStartSide|     stadiumName|stadiumLength|stadiumWidth|possessionTeam|hom

In [25]:
# variável que indica o time com a posse
df_games_events.groupBy('homeTeam').count().show()

+--------+------+
|homeTeam| count|
+--------+------+
|    NULL|  6918|
|    true|472671|
|   false|465565|
+--------+------+



In [26]:
df_games_events = df_games_events.dropna(subset='homeTeam') #dropando eventos onde nenhum dos dois times tem a posse

## Normalização do campo

In [27]:
# criar atacantes, defensores e direção do ataque
# queremos atacantes sempre em direção à direita
# time com a posse está atacando e time sem está defendendo

df_possesion_tracking_events_sides = (
    df_games_events
    .select(
        'eventId', 
        'homeTeam', 
        'homePlayers_parsed', 
        'awayPlayers_parsed', 
        'balls_parsed', 
        'homeTeamAttackDirection', 
        'awayTeamAttackDirection'
        )
    # time atacando = se o time da casa tiver a posse, pega tracking home, se não pega tracking away
    .withColumn(
        "attackingPlayers",
        F.when(F.col("homeTeam"), F.col("homePlayers_parsed"))
        .otherwise(F.col("awayPlayers_parsed"))
    )
    # time defendendo = se o time da casa tiver a posse, pega tracking away, se não pega tracking home
    .withColumn(
        "defendingPlayers",
        F.when(F.col("homeTeam"), F.col("awayPlayers_parsed"))
        .otherwise(F.col("homePlayers_parsed"))
    )
    .withColumn(
        "attackingDirection",
        F.when(F.col("homeTeam"), F.col("homeTeamAttackDirection"))
        .otherwise(F.col("awayTeamAttackDirection"))
    )
    # flag para normalização (para tratar ataque sempre pra direita)
    .withColumn(
        "is_flipped",
        F.col("attackingDirection") == "Left"
    )
)

df_possesion_tracking_events_sides.show()

+--------------------+--------+--------------------+--------------------+--------------------+-----------------------+-----------------------+--------------------+--------------------+------------------+----------+
|             eventId|homeTeam|  homePlayers_parsed|  awayPlayers_parsed|        balls_parsed|homeTeamAttackDirection|awayTeamAttackDirection|    attackingPlayers|    defendingPlayers|attackingDirection|is_flipped|
+--------------------+--------+--------------------+--------------------+--------------------+-----------------------+-----------------------+--------------------+--------------------+------------------+----------+
|9e6a498f54bad910e...|   false|[{25.001, 15.746,...|[{4.63, -17.929, ...|[{0.3, -0.13, -4....|                   Left|                  Right|[{4.63, -17.929, ...|[{25.001, 15.746,...|             Right|     false|
|60717c0b4d7bb1d57...|   false|[{25.001, 15.746,...|[{4.63, -17.929, ...|[{0.3, -0.13, -4....|                   Left|                  Righ

In [28]:
players_tracking_norm = (
        lambda p: F.struct(
            F.when(F.col("is_flipped"), -p["x"])
            .otherwise(p["x"])
            .alias("x"),
            
            F.when(F.col("is_flipped"), -p["y"])
            .otherwise(p["y"])
            .alias("y"),
            
            p["player"].alias("player"),
            p["visibility"].alias("visibility"),
            p["confidence"].alias("confidence")
        )
)

players_tracking_norm

balls_norm = (
    F.transform(
        "balls_parsed",
        lambda b: F.struct(
            b["z"].alias("z"),
            b["y"].alias("y"),
            F.when(F.col("is_flipped"), -b["x"])
            .otherwise(b["x"])
            .alias("x"),
            b["visibility"].alias("visibility")
        )
    )
)

# df para normalizar os atacantes, defensores e bola sempre atacando do lado direito
df_possesion_tracking_events_sides_norm = (
    df_possesion_tracking_events_sides
    # normalizar atacantes
    .withColumns({
        "attackingPlayersNorm": F.transform("attackingPlayers", players_tracking_norm),
        # normalizar defensores
        "defendingPlayersNorm": F.transform("defendingPlayers", players_tracking_norm),
        # normalizar a bola
        "ballsNorm": balls_norm
    })
)

In [29]:
df_possesion_tracking_events_sides_norm.show()

+--------------------+--------+--------------------+--------------------+--------------------+-----------------------+-----------------------+--------------------+--------------------+------------------+----------+--------------------+--------------------+--------------------+
|             eventId|homeTeam|  homePlayers_parsed|  awayPlayers_parsed|        balls_parsed|homeTeamAttackDirection|awayTeamAttackDirection|    attackingPlayers|    defendingPlayers|attackingDirection|is_flipped|attackingPlayersNorm|defendingPlayersNorm|           ballsNorm|
+--------------------+--------+--------------------+--------------------+--------------------+-----------------------+-----------------------+--------------------+--------------------+------------------+----------+--------------------+--------------------+--------------------+
|9e6a498f54bad910e...|   false|[{25.001, 15.746,...|[{4.63, -17.929, ...|[{0.3, -0.13, -4....|                   Left|                  Right|[{4.63, -17.929, ...|[{2

In [30]:
df_possesion_tracking_events_sides_norm.select('is_flipped', 'balls_parsed', 'ballsNorm').show(truncate=False)

+----------+---------------------------------+----------------------------------+
|is_flipped|balls_parsed                     |ballsNorm                         |
+----------+---------------------------------+----------------------------------+
|false     |[{0.3, -0.13, -4.7, ESTIMATED}]  |[{0.3, -0.13, -4.7, ESTIMATED}]   |
|false     |[{0.3, -0.13, -4.7, ESTIMATED}]  |[{0.3, -0.13, -4.7, ESTIMATED}]   |
|false     |[{0.0, 2.06, -20.53, VISIBLE}]   |[{0.0, 2.06, -20.53, VISIBLE}]    |
|false     |[{0.0, 2.06, -20.53, VISIBLE}]   |[{0.0, 2.06, -20.53, VISIBLE}]    |
|false     |[{0.02, -27.81, -6.76, VISIBLE}] |[{0.02, -27.81, -6.76, VISIBLE}]  |
|false     |[{0.02, -27.81, -6.76, VISIBLE}] |[{0.02, -27.81, -6.76, VISIBLE}]  |
|true      |[{0.81, -2.02, 44.72, ESTIMATED}]|[{0.81, -2.02, -44.72, ESTIMATED}]|
|true      |[{0.81, -2.02, 44.72, ESTIMATED}]|[{0.81, -2.02, -44.72, ESTIMATED}]|
|true      |[{0.06, -13.1, 42.42, ESTIMATED}]|[{0.06, -13.1, -42.42, ESTIMATED}]|
|true      |[{0.

In [31]:
df_possesion_tracking_events_sides_norm.select('is_flipped', 'attackingPlayers', 'defendingPlayers', 'attackingPlayersNorm', 'defendingPlayersNorm').show()

+----------+--------------------+--------------------+--------------------+--------------------+
|is_flipped|    attackingPlayers|    defendingPlayers|attackingPlayersNorm|defendingPlayersNorm|
+----------+--------------------+--------------------+--------------------+--------------------+
|     false|[{4.63, -17.929, ...|[{25.001, 15.746,...|[{-17.929, 4.63, ...|[{15.746, 25.001,...|
|     false|[{4.63, -17.929, ...|[{25.001, 15.746,...|[{-17.929, 4.63, ...|[{15.746, 25.001,...|
|     false|[{2.954, -20.167,...|[{21.9, 11.349, {...|[{-20.167, 2.954,...|[{11.349, 21.9, {...|
|     false|[{2.954, -20.167,...|[{21.9, 11.349, {...|[{-20.167, 2.954,...|[{11.349, 21.9, {...|
|     false|[{5.275, -24.306,...|[{5.468, 3.131, {...|[{-24.306, 5.275,...|[{3.131, 5.468, {...|
|     false|[{5.275, -24.306,...|[{5.468, 3.131, {...|[{-24.306, 5.275,...|[{3.131, 5.468, {...|
|      true|[{24.01, 22.353, ...|[{-0.484, -0.717,...|[{-22.353, -24.01...|[{0.717, 0.484, {...|
|      true|[{24.01, 22.353, .

In [ ]:
# validar se a normalização de lado esta certa com ataque sempre pra direita (dicas no chatGPT)  
# conforme validado, criar as 3 variáveis de ameaça

In [ ]:
# df_ball.select('is_flipped', 'balls_parsed', 'ballsNorm').show(truncate=False)

In [ ]:
# event_id

# possessionTeam

# attackingPlayersNorm
# defendingPlayersNorm

# ball_x_norm
# ball_y_norm

# Ataque ---> Direita

# Gol defendido = (-L/2, 0)

# Gol atacado = (+L/2, 0)

In [ ]:
# event_id-season-gameId-match_id-possession_team-attacking_team-defending_team-ball_x-ball_y-attacking_players-defending_players-threat-threat_delta

## Domínios

### Tipos de eventos:

- FIRSTKICKOFF: Inicio do primeiro tempo
- SECONDKICKOFF: Inicio do segundo tempo
- TC: Touch
- RE: Rebound
- BC: Ball Carry
- CL: Clearance
- CR: Cross
- CH: Challenge 
- OTB: A possession with a player on the ball
- PA: Pass
- FO: Foul
- FOUL: Additional foul
- SH: Shot

### Domínio: Desempenho Técnico Defensivo

### Eventos que queremos (Defensivos):

- Ofensivos como CR, PA e SH queremos que o resultado dele seja uma interferência da defesa adversária
- Defensivos como CL, CH, FO queremos que o tipo seja ação defensiva

- CL: Clearance
    - Qualquer CLEARANCE_OUTCOME_TYPE (A,B,D,E,O,P,S,U)
    - obs: talvez não E e U pq são FairPlay
    - obs2: P - Player e S - Stoppage não sei oq sejam, mas vou deixar

- CR: Cross
    - CROSS_OUTCOME_TYPE:
    - B - Blocked
    - D - Defensive Interception

- CH: Challenge. 
    - CHALLENGE_TYPE:
    - ‘5’ - 50/50. This is a duel type where two players compete for a loose ball.
    - A - Aerial duel. As the name suggests a duel type similar to 50-50, but with the ball coming from above.
    - B - Tackle from behind. As the name suggests a tackle attempt where the carrier puts their body in between the ball and the challenger as the tackle is attempted.
    - D - Dribble. The player tries to take on a defender in an attempt to get past them.
    - G - Goalkeeper smothers ball. A duel between the goalkeeper and a line player where the ball is loose and the goalkeeper tries to capture the ball.
    - H - Shielding. Similar to tackle from behind, but on a shielding challenge the carrier actively shields a defender who does not attempt a tackle
    - K - Hand tackle by goalkeeper. Despite the name, it is a duel type similar to goalkeeper smothers, but the keeper tries to parry the ball rather than retain it.
    - L - Slide tackle. Tackle type where the challenger slides to attempt to win the ball. Note that a player could be sliding on a dribble or 50-50, to be classed as a slide tackle it needs to be first and foremost a tackle.
    - S - Shoulder to shoulder. Tackle type where the challenger tries to win the ball with physical contact initiated with the body.
    - T - Standing tackle. Tackle attempt, usually from the front or side, that does not fit the other tackle types 
    - OBS1: **Único que não entraria como AD aqui seria o 'D'.**
    - OBS2: **Não estamos considerando outcome dos eventos.**

- PA: Pass
    - PASS_OUTCOME_TYPE:
    - B - Blocked
    - D - Defensive Interception

- FO: Foul
    - qualquer FOUL_TYPE = A, I, M
    - não vi evento de penalti, então teria q pegar a região dentro da area e evento de falta marcado ali (FOUL_TYPE == I)

- FOUL: Additional foul
    - são faltas adicionais no mesmo lance divida em mais de um evento, mas nos dados fica tudo NULL, então n vou add. FO já tem o evento principal de falta

- SH: Shot
    - SHOT_OUTCOME_TYPE:
    - B - Block on target. (Ball was going on target, but got blocked)
    - C - Block off target. (Ball was going off target, and got blocked)
    - F - Save off target. (Ball was going off target when it got saved)
    - L - Goalline clearance. (Ball is past the goalkeeper and a defender stops it from going into the net)
    - S - Save on target. (Ball was going on target and got saved).

### Domínio: Ameaça

Vamos criar as variáveis de ameaça para os dois times, respeitando os sentidos de ataques deles. Isso apenas para os eventos defensivos.

In [44]:
# eventos com posse pega só do time com a posse
# eventos sem posse, pega dos dois e faz a média

#### Métrica 1: Distância percorrida no campo (medida pela menor distância entre os escanteios)

#### Métrica 2: Quantidade total de jogadores dos dois times (entre a bola e o gol)

#### Métrica 3: Diferencial da quantidade de defensores e atacantes